# MLP: Classification with PyTorch - Iris Dataset

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load diabetes dataset from sklearn
diabetes = load_iris()

# Info about the dataset
print(diabetes.DESCR)

In [ ]:
# Convert to pandas DataFrame for easier manipulation
df = pd.DataFrame(data=diabetes.data, columns=diabetes.feature_names)
df['target'] = diabetes.target

# Display the first few rows of the dataset
display(df.head())

In [ ]:
# Boxplot for each feature
df.hist(bins=20, figsize=(12, 10))

In [ ]:
# Boxplot for each feature
df.iloc[:, :-1].boxplot()

In [ ]:
# Extract features and target variable
_X = df.iloc[:, :-1].values
print(_X.shape)

In [ ]:
# One-hot encode the target variable in classification problems
_Y = pd.get_dummies(df['target']).astype(float).values
print(_Y.shape)

# Verify the range of the features before scaling
print(df['target'].iloc[:5])
print(_Y[:5])

In [ ]:
# Split the dataset into training and testing sets
_X_train, _X_test, _Y_train, _Y_test = train_test_split(
    _X, _Y, test_size=0.3, random_state=0
)
print(_X_train.shape)
print(_X_test.shape)
print(_Y_train.shape)
print(_Y_test.shape)

In [ ]:
# Scale the features
scX = StandardScaler()
X_train = scX.fit_transform(_X_train)
X_test = scX.transform(_X_test)

# No need to scale the target variable for classification problems
Y_train = _Y_train
Y_test = _Y_test

In [ ]:
# Verify the range of the features after scaling
pd.DataFrame(X_train).boxplot()
# pd.DataFrame(X_test).boxplot()

In [ ]:
# Convert numpy arrays to PyTorch tensors
X_train_pt = torch.from_numpy(X_train).float()
X_test_pt = torch.from_numpy(X_test).float()
Y_train_pt = torch.from_numpy(Y_train).float()
Y_test_pt = torch.from_numpy(Y_test).float()

In [ ]:
# Define the model
num_features = _X.shape[1]
num_outputs = _Y.shape[1]


class MyModel(nn.Module):
    def __init__(self, num_features, num_outputs):
        super(MyModel, self).__init__()
        self.fc1 = nn.Linear(num_features, 24)
        self.fc2 = nn.Linear(24, 12)
        self.fc3 = nn.Linear(12, 6)
        self.fc4 = nn.Linear(6, num_outputs)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)
        return x


model = MyModel(num_features, num_outputs)

In [ ]:
# Model summary
from torchinfo import summary

summary(model, input_size=(1, num_features))

In [ ]:
n_epochs = 500  # number of epochs to run

optimizer = torch.optim.Adam(
    model.parameters(), lr=0.01
)  # Adam optimizer with learning rate 0.01
loss_fn = nn.CrossEntropyLoss()  # Cross-entropy loss for multi-class classification

loss_arr = []
acc_arr = []
for epoch in range(n_epochs):
    # Training Phase
    model.train()  # set model to training mode
    optimizer.zero_grad()  # reset the gradients before backward pass
    Y_pred = model(X_train_pt)  # forward pass - get predictions for training data
    loss = loss_fn(
        Y_pred, Y_train_pt
    )  # compute MSE loss between predictions and labels

    # Backward pass
    loss.backward()  # compute gradients

    # Update weights
    optimizer.step()  # update the weights

    epoch_train_loss = loss.item()  # extract numerical loss value
    loss_arr.append(epoch_train_loss)  # record loss for this epoch

    # Compute accuracy for training set
    predicted_classes = torch.argmax(Y_pred, dim=1)
    true_classes = torch.argmax(Y_train_pt, dim=1)
    accuracy = (predicted_classes == true_classes).sum().item() / len(true_classes)
    acc_arr.append(accuracy)
    
# Evaluation / Test phase (no grad needed)
with torch.no_grad():
    test_pred = model(X_test_pt)  # get predictions on test set
    final_loss = loss_fn(test_pred, Y_test_pt)  # compute final test loss
    
    # Compute accuracy for test set
    predicted_classes = torch.argmax(test_pred, dim=1)
    true_classes = torch.argmax(Y_test_pt, dim=1)
    test_accuracy = (predicted_classes == true_classes).sum().item() / len(true_classes)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.lineplot(loss_arr, ax=axes[0])  # plot loss over epochs using seaborn
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross Entropy Loss")
axes[0].set_title(f"Final Loss = {final_loss}")

# Plot accuracy over epochs
sns.lineplot(acc_arr, ax=axes[1])
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title(f"Final Accuracy = {test_accuracy}")    